# scene01 proof of concept: changes, uncertainty and robot heat maps (Colab, one T4)

Research paths P1 (shared reconstruction) and P2 (change detection), with the P3-P6 outputs layered on top.
Branch `exp/change/poc-lowcompute`. Runs until the session ends; every finished step is pushed to GitHub and
skipped by later sessions (any Google account), so just **Run all** again to continue.

**Method (low compute, no training):** VGGT-1B reconstructs frames from both recordings jointly in one shared 3D
frame (~1-2 min per pair on a T4). Every point is projected into the other recording's cameras: it is *gone* if
they see through it, *occluded* if something is in front, *appearance-changed* if the surface agrees but DINOv2
features differ. Thresholds come from comparing views of the **same** recording (nothing changed there), so no
labels are used. Changed points are clustered into objects, re-identified across recordings (moved = removed +
added that look alike) and labelled with CLIP.

**Per pair it writes** (`results/change/<RUN_ID>/<step>/`):
- `heatmaps.png`: surface uncertainty (cm), surface shift / deformation (cm), height, robot reachability,
  gripper approachability, interactability
- `base_placement.png`: where on the floor a robot could stand and how much of the scene (and of the changes) it reaches
- `change_map.png`, `overlays.png`, `crops.png`, `calibration.png`
- `viewer.html`: interactive 3D with toggleable change and heat-map layers (download and open in a browser)
- `scene_report.md`: changes with metric sizes and uncertainty, robot section, simulator randomisation ranges
- `cloud.ply`, `fields.ply`, `metrics.json`, `changes.json`, `report.md`
- `summary_*` steps: one table and figure across all pairs (false changes on static/lighting pairs vs real changes)

**Setup:** Runtime -> Change runtime type -> **T4 GPU**. Secrets panel (key icon): `GITHUB_TOKEN` (fine-grained,
Contents read/write on UncertainScene) with notebook access on. Never paste a token into a cell.

Metric scale assumes the phone was held at 1.35 m (range 1.1-1.6 m, carried as uncertainty).

In [ ]:
# ---- Config ---------------------------------------------------------------------------
BRANCH = "exp/change/poc-lowcompute"
RUN_ID = "2026-10-10_poc-scene01"   # keep the same id to resume; change it only for a fresh run
PATH_NAME = "change"
EXPECTED_COMMIT = ""                # optional: short sha you expect; setup stops if HEAD differs
RETRY_FAILED = False                # True re-runs steps that failed with an error (after a code fix)
ONLY_PHASES = None                  # e.g. ["p1", "p2"] to restrict; None = the whole queue
SHOW_FIGURES = True                 # show heat maps inline as steps finish
REPO = "FinPThomas/UncertainScene"

In [ ]:
# ---- Setup: every check logged, token never printed ---------------------------------------
import base64, datetime, os, subprocess, sys, time
from pathlib import Path

SETUP_LOG = []
def say(msg, level="INFO"):
    line = f"{datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d %H:%M:%S}Z {level:5s} {msg}"
    SETUP_LOG.append(line)
    print(line, flush=True)

def fail(msg, hint=""):
    say(msg + (f"\n      fix: {hint}" if hint else ""), "ERROR")
    raise SystemExit(msg)

def secret(name):
    """Returns (value, where it came from). Tries Colab secrets, Kaggle secrets, environment."""
    try:
        from google.colab import userdata
        try:
            v = userdata.get(name)
            if v:
                return v, "Colab secrets"
        except Exception as e:  # SecretNotFoundError / NotebookAccessError
            say(f"Colab secret {name}: {type(e).__name__}", "WARN")
    except ImportError:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v, "Kaggle secrets"
    except Exception:
        pass
    v = os.environ.get(name)
    return (v, "environment") if v else (None, None)

say(f"[1/8] Python {sys.version.split()[0]}, platform: " + ("Kaggle" if os.path.isdir("/kaggle") else "Colab" if os.path.isdir("/content") else "local"))

# ---- 2. token present and well formed
token, where = secret("GITHUB_TOKEN")
if not token:
    fail("[2/8] GITHUB_TOKEN not found",
         "Colab: key icon in the left bar -> Add new secret named GITHUB_TOKEN, paste the token, switch 'Notebook access' on, then re-run this cell")
token = token.strip()
kind = ("fine-grained (github_pat_)" if token.startswith("github_pat_") else "classic (ghp_)" if token.startswith("ghp_")
        else "unrecognised format")
say(f"[2/8] GITHUB_TOKEN found in {where}: {kind}, {len(token)} characters (value never printed)")
if kind == "unrecognised format":
    say("      the token does not start with github_pat_ or ghp_; check it was pasted whole", "WARN")
os.environ["GITHUB_TOKEN"] = token
label, _ = secret("UNCERTAINSCENE_ACCOUNT")
if label:
    os.environ["UNCERTAINSCENE_ACCOUNT"] = label
    say(f"      session label from UNCERTAINSCENE_ACCOUNT: {label}")

# ---- 3. token valid, who it belongs to, when it expires
import requests
H = {"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"}
try:
    r = requests.get("https://api.github.com/user", headers=H, timeout=20)
except requests.RequestException as e:
    fail(f"[3/8] cannot reach api.github.com: {type(e).__name__}", "check the runtime has internet (Kaggle: Settings -> Internet on)")
if r.status_code == 401:
    fail("[3/8] GitHub rejected the token (401: invalid, expired or revoked)", "create a new fine-grained token and update the secret")
if r.status_code != 200:
    fail(f"[3/8] GitHub /user returned {r.status_code}: {r.text[:200]}")
exp = r.headers.get("github-authentication-token-expiration", "no expiry date reported")
say(f"[3/8] token valid for GitHub user '{r.json().get('login')}'; expires: {exp}; API calls left this hour: {r.headers.get('x-ratelimit-remaining')}")

# ---- 4. token can see the repository and the branch
r = requests.get(f"https://api.github.com/repos/{REPO}", headers=H, timeout=20)
if r.status_code == 404:
    fail(f"[4/8] token cannot see {REPO} (404)", "fine-grained token: Repository access -> Only select repositories -> add UncertainScene")
if r.status_code != 200:
    fail(f"[4/8] /repos/{REPO} returned {r.status_code}: {r.text[:200]}")
say(f"[4/8] repository {REPO} visible (private: {r.json().get('private')})")
r = requests.get(f"https://api.github.com/repos/{REPO}/branches/{BRANCH}", headers=H, timeout=20)
if r.status_code != 200:
    fail(f"[4/8] branch {BRANCH} not found ({r.status_code})", "check BRANCH in the config cell")
say(f"      branch {BRANCH} at {r.json()['commit']['sha'][:7]}")

# ---- 5. clone (token passed per command, never written to .git/config)
ROOT = Path("/kaggle/working" if os.path.isdir("/kaggle/working") else "/content")
REPO_DIR = ROOT / "UncertainScene"
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
AUTH = ["-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {auth}"]

def git(*args, cwd=REPO_DIR):
    p = subprocess.run(["git", *AUTH, *args], cwd=cwd, capture_output=True, text=True)
    if p.returncode:
        raise RuntimeError(p.stderr.replace(token, "***").replace(auth, "***"))
    return p.stdout.strip()

t0 = time.time()
if not (REPO_DIR / ".git").exists():
    git("clone", "--branch", BRANCH, f"https://github.com/{REPO}.git", str(REPO_DIR), cwd=ROOT)
else:
    git("fetch", "origin", BRANCH)
    git("checkout", "-B", BRANCH, f"origin/{BRANCH}")
head = git("rev-parse", "--short", "HEAD")
say(f"[5/8] code at {head} ({git('log', '-1', '--format=%s')}) in {time.time() - t0:.0f} s")
if EXPECTED_COMMIT and not head.startswith(EXPECTED_COMMIT[:7]):
    fail(f"stale code: HEAD is {head}, expected {EXPECTED_COMMIT}", "clear EXPECTED_COMMIT or wait for the push")

# ---- 6. dependencies
t0 = time.time()
pip = [sys.executable, "-m", "pip", "install", "-q"]
for cmd in (pip + ["-e", f"{REPO_DIR}[poc]"], pip + ["--no-deps", "git+https://github.com/facebookresearch/vggt.git"]):
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode:
        fail(f"[6/8] pip failed: {p.stderr[-800:]}")
sys.path.insert(0, str(REPO_DIR / "src"))
import torch, vggt  # noqa: F401
say(f"[6/8] dependencies installed in {time.time() - t0:.0f} s (torch {torch.__version__})")

# ---- 7. write access: a dry-run push proves the token can push to this branch
from uncertainscene.infra.publish import Publisher
pub = Publisher(REPO_DIR, BRANCH, token)
pub.ensure_identity()
try:
    pub.check_push_access()
except Exception as e:
    fail(f"[7/8] token cannot push to {BRANCH}: {str(e).replace(token, '***')[:300]}",
         "fine-grained token: Repository permissions -> Contents -> Read and write")
say(f"[7/8] push access to {BRANCH} OK")

In [ ]:
# ---- 8. GPU, memory guardrails and run state ---------------------------------------------
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
if not torch.cuda.is_available():
    fail("[8/8] no GPU", "Runtime -> Change runtime type -> T4 GPU, then Run all again")
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

from uncertainscene.infra import Run
from uncertainscene.infra.guard import ResourceLimit, read_usage
from uncertainscene.infra.store import Store, repo_slug
from uncertainscene.recon.vggt import max_frames_for_gpu

run = Run.from_env(run_id=RUN_ID, path=PATH_NAME, branch=BRANCH)
b = run.budget
say(f"[8/8] GPU {torch.cuda.get_device_name(0)} {gpu_gb:.1f} GB: fits ~{max_frames_for_gpu()} frames per reconstruction "
    "(more are capped; CUDA out-of-memory retries with 20% fewer)")
say(f"      machine now: {read_usage(b.work_dir).line()}")
say(f"      guardrails: RAM warn {b.ram_soft_frac:.0%} / stop {b.ram_hard_frac:.0%}, disk warn {b.disk_soft_free_gb} GB / "
    f"stop {b.disk_hard_free_gb} GB free, checked every {b.monitor_interval_s} s and between every stage of a step; "
    f"a step only starts if its RAM, disk and time fit; session plan {b.session_hours} h minus {b.safety_minutes} min")
say(f"      session label {run.account}; run state:\n" + run.state.summary())

CACHE = Path(b.work_dir) / "cache"      # reconstructions reused across steps (outside step scratch)
RUN_DIR = REPO_DIR / "results" / PATH_NAME / RUN_ID
try:
    store = Store(repo_slug(REPO_DIR))            # dense clouds as release assets (best effort)
except Exception as e:
    store = None
    say(f"no release store ({type(e).__name__}); dense clouds stay local", "WARN")

# push the setup log so a failed or odd session can be diagnosed from GitHub
log_rel = Path("results") / PATH_NAME / RUN_ID / "logs" / f"setup_{run.account}.txt"
(REPO_DIR / log_rel).parent.mkdir(parents=True, exist_ok=True)
(REPO_DIR / log_rel).write_text("\n".join(SETUP_LOG) + "\n")
try:
    pub.publish([log_rel], f"results({PATH_NAME}): setup log {RUN_ID} ({run.account})", fallback_branch=f"results/{RUN_ID}")
    say(f"setup log pushed: {log_rel}")
except Exception as e:
    say(f"could not push the setup log: {str(e).replace(token, '***')[:200]}", "WARN")

In [ ]:
# ---- The queue: runs until the session ends ---------------------------------------------
# p1 proof of concept with every heat map -> p1b light vs dark contrasts -> p2 dense versions ->
# p3 robot scenarios -> p4 two sanity ablations -> p5 other take combinations -> p6 fewer frames ->
# p7 repeats with other frames. Each step: skipped if done (by any account) or claimed by a live
# session elsewhere; not started if the RAM, disk or session time it needs is not available.
# Every step pushes its own step_log.txt with stage timings and memory use.
import gc
from IPython.display import Image, Markdown, display
from uncertainscene.eval import summarize
from uncertainscene.experiments.poc import queue, step_fn, trim_cache

def summary_step(out, work):
    summarize.write(RUN_DIR, out)
    return f"summary of {len(summarize.load(RUN_DIR))} steps (summary.md, lighting.md)"

counts, t_start = {}, time.time()
try:
    for item in queue():
        name = item if isinstance(item, str) else item.name
        if ONLY_PHASES and name.split("_")[0] not in ONLY_PHASES and not name.startswith("summary_"):
            continue
        t0 = time.time()
        if isinstance(item, str):
            status = run.step(item, summary_step, needs=dict(disk_gb=0.2, ram_gb=1, minutes=2), retry_failed=RETRY_FAILED)
        else:
            status = run.step(item.name, step_fn(item, CACHE, store, run.guard.check), needs=item.needs,
                              retry_failed=RETRY_FAILED)
            trim_cache(CACHE, max_gb=6)
            gc.collect()
            torch.cuda.empty_cache()
        counts[status] = counts.get(status, 0) + 1
        if status not in ("skipped",):
            say(f"{name}: {status} in {(time.time() - t0) / 60:.1f} min | {run.state.step(name).note or ''} | "
                f"{read_usage(b.work_dir).line()} | {run.guard.minutes_left():.0f} min left")
        if status == "failed":
            say(f"      see results/{PATH_NAME}/{RUN_ID}/{name}/error.txt and step_log.txt", "WARN")
        hm = run.out_dir(name) / "heatmaps.png"
        if SHOW_FIGURES and status == "done" and not isinstance(item, str) and item.outputs == "full" and hm.exists():
            display(Markdown(f"### {name}\n{run.state.step(name).note}"), Image(str(hm), width=1100))
except ResourceLimit as e:
    say(f"stopping cleanly, everything finished so far is pushed: {e}", "WARN")
say(f"session: {counts} in {(time.time() - t_start) / 3600:.1f} h")

In [ ]:
# ---- Finish: refresh the overall summary and show the headline ---------------------------
try:
    run.step(f"summary_session_{run.account}", summary_step, needs=dict(disk_gb=0.2, ram_gb=1, minutes=2))
except ResourceLimit as e:
    print("No time left for the final summary:", e)
run.close()
print(run.state.summary())
s = run.out_dir(f"summary_session_{run.account}") / "summary.md"
if s.exists():
    display(Markdown(s.read_text()[:6000]))